# Reporting

In [51]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

import sys 
sys.path.append('..')  

from module.dataload import DPN_data
import ymlconfig

%matplotlib inline
%load_ext autoreload
%autoreload 2

warnings.filterwarnings('ignore')
np.set_printoptions(precision=3)  # decimal places for outputs from numpy
pd.set_option("display.precision", 3)  # decimal places for outputs from pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Read Counterfactuals Config File

In [52]:
config_path = Path(r'experiments')
config_filename =  "bin_cf_dev.yml"
config_dict = ymlconfig.load_config(config_path / config_filename)
config = ymlconfig.dict_to_namespace(config_dict)
config_dict

{'experiment': {'summary': 'binary classification - Counterfactual Analysis (Dev)',
  'classification_type': 'binary',
  'stage': 'counterfactuals',
  'tag': 'development',
  'verbosity': 1,
  'random_seed': 42},
 'data': {'dataset_path': '../dataset/Sudoscan Working File with Stats.xlsx'},
 'model': {'code': 'catboost', 'name': 'CatBoost'},
 'explainability': {'ksplit_trained_model_results_file': 'binary\\explainability\\catboost\\final\\catboost_ksplit_trained_models.joblib',
  'rundate': '2026-03-18',
  'tag': 'final'},
 'dice': {'method': 'genetic',
  'global_cf': {'total_CFs': 10, 'posthoc_sparsity_param': 'None'},
  'local_cf': {'nrepeats': 2,
   'total_CFs': 20,
   'posthoc_sparsity_algorithm': 'binary',
   'posthoc_sparsity_param': 0.05,
   'proximity_weight': 0.5,
   'diversity_weight': 1.0,
   'categorical_penalty': 0.1,
   'algorithm': 'DiverseCF'},
  'sufficiency': {'maxiterations': 200},
  'necessity': {'maxiterations': 500, 'total_CFs': 2, 'nrepeats': 5}},
 'reporting': {

## Set Directories

In [53]:
nsplits = 3

outputdir = config_path /  config.experiment.classification_type /  config.experiment.stage / config.model.code / config.experiment.tag 
assert outputdir.is_dir()

split_output_dirs = []
for midx in range(nsplits):
    s = outputdir / f'split{midx}'
    assert s.is_dir()
    split_output_dirs.append(s)

## Instances of Interest

In [54]:
outputdir, split_output_dirs

(WindowsPath('experiments/binary/counterfactuals/catboost/development'),
 [WindowsPath('experiments/binary/counterfactuals/catboost/development/split0'),
  WindowsPath('experiments/binary/counterfactuals/catboost/development/split1'),
  WindowsPath('experiments/binary/counterfactuals/catboost/development/split2')])

In [55]:
ioi = {}
for midx in range(nsplits):
    filename = f'{config.model.code}_split{midx}_instances_of_interest.csv'
    df = pd.read_csv(split_output_dirs[midx] / filename)
    ioi[midx] = {
        'Model': f'Split {midx}', 
        'Instances': df.shape[0], 
        'Misclassified': df.misclassified.sum()
        }
ioi

{0: {'Model': 'Split 0', 'Instances': 6, 'Misclassified': 4},
 1: {'Model': 'Split 1', 'Instances': 7, 'Misclassified': 4},
 2: {'Model': 'Split 2', 'Instances': 9, 'Misclassified': 6}}

In [56]:
ioi_df = pd.DataFrame(ioi).T
ioi_df

,Model,Instances,Misclassified
0,Split 0,6,4
1,Split 1,7,4
2,Split 2,9,6


## Model 0 Summary

In [57]:
midx = 0
ioi_filename = f'{config.model.code}_split{midx}_instances_of_interest.csv'
ioi_df = pd.read_csv(split_output_dirs[midx] / ioi_filename)
ioi_df = ioi_df.rename(columns={'Unnamed: 0':'ID'})
ioi_df

,ID,SEX,AGE,SUBJ,DM_DUR,INSULIN,HBA1C,HPN,PAOD,DSLPDMIA,CKD,GBS,DEC_VS,DEC_PPS,DEC_LTS,DEC_AR,MNSI,SSA_L,SSC_L,SPSA_L,SPSC_L,MCV_L,DL_L,CMAPANK_L,CMAPKNE_L,FWAVE_L,SSA_R,SSC_R,SPSA_R,SPSC_R,MCV_R,DL_R,CMAPANK_R,CMAPKNE_R,FWAVE_R,FEET_MEAN_ESC,FEET_PCT_ASYM,HAND_MEAN_ESC,HAND_PCT_ASYM,NS,CAS,pred_proba,margin,pred,actual,misclassified
0,4,1,57.0,0,5.0,1.0,14.4,0,0,0,0,0,1,0,1,1,1.0,4.19,41.90,3.70,38.20,38.5,4.50,8.89,6.88,48.3,5.36,45.5,4.42,39.50,38.3,4.00,10.29,8.82,49.9,54.0,3.0,63.0,0.0,54.0,36.0,0.719,0.083,1,1,False
1,55,0,67.0,1,11.0,0.0,11.1,1,0,1,0,0,1,1,1,1,4.0,8.74,47.50,4.03,44.50,45.2,4.85,14.13,9.15,49.2,7.24,46.5,7.04,53.10,42.5,4.00,8.42,6.23,50.0,67.0,2.0,49.0,9.0,47.0,32.0,0.767,0.131,1,1,False
2,67,1,56.0,1,0.0,0.0,5.1,0,0,0,0,0,0,0,0,0,3.0,16.27,52.00,12.16,50.70,46.6,3.75,9.46,6.93,47.0,13.80,50.0,13.48,53.00,43.8,3.20,7.41,5.61,47.7,71.0,8.0,63.0,6.0,60.0,30.0,0.187,0.449,0,1,True
3,106,1,33.0,1,3.0,1.0,9.0,0,0,0,0,0,0,1,0,1,3.0,4.88,49.20,0.00,0.00,43.2,3.50,13.14,10.75,43.3,2.68,45.3,0.00,0.00,44.0,3.45,9.98,7.18,42.4,76.0,0.0,76.0,12.0,90.0,4.0,0.524,0.111,0,1,True
4,128,0,74.0,1,30.0,0.0,4.9,1,0,1,0,0,0,1,1,1,4.0,14.40,47.60,11.03,45.50,45.1,3.95,10.29,7.78,46.4,7.45,46.4,11.65,48.50,42.3,3.45,7.71,5.12,46.6,46.0,0.0,60.0,0.0,34.0,30.0,0.275,0.360,0,1,True
5,158,0,43.0,1,2.0,0.0,5.2,0,0,1,0,0,0,0,1,0,9.0,12.19,1.96,9.78,2.38,47.2,4.15,14.75,8.91,43.3,34.31,57.7,19.70,2.58,49.3,3.45,15.36,9.43,42.9,57.0,6.0,50.0,7.0,70.0,29.0,0.037,0.598,0,1,True


In [58]:
cf = {}
for qidx in ioi_df.ID.values:
    qidx_filename = f'{config.model.code}_split{midx}_local_cf.csv'
    qidx_df = pd.read_csv(split_output_dirs[midx] / 'filtered_progressive' / str(qidx).zfill(3) / qidx_filename)
    cf_count = qidx_df.shape[0]    
    cf[qidx] = {
        'Patient Index': qidx, 
        'CF Count': cf_count, 
        'Sparsity': np.nan if not cf_count else qidx_df['sparsity'].mean(),
        'L1 Mean': np.nan if not cf_count else qidx_df['L1_dist'].mean(),
        'L2 Mean': np.nan if not cf_count else qidx_df['L2_dist'].mean(),
        }
cf_df = pd.DataFrame(cf).T.reset_index(drop=True)
cf_df['Patient Index'] = cf_df['Patient Index'].astype(int)
cf_df['CF Count'] = cf_df['CF Count'].astype(int)
cf_df

,Patient Index,CF Count,Sparsity,L1 Mean,L2 Mean
0,4,1,29.000,135.270,33.364
1,55,0,NaN,NaN,NaN
2,67,22,22.636,95.595,31.052
3,106,0,NaN,NaN,NaN
4,128,7,19.857,94.031,28.886
5,158,13,23.308,103.557,29.989


## Full Script

In [59]:
ioi_summary_dfs = []
gi_dfs = []
cf_dfs = {}
for midx in range(nsplits):
    ioi = {}

    # Read Instance of Interest
    ioi_filename = f'{config.model.code}_split{midx}_instances_of_interest.csv'
    ioi_df = pd.read_csv(split_output_dirs[midx] / ioi_filename)
    ioi_df = ioi_df.rename(columns={'Unnamed: 0':'ID'})
    
    # Read Global Importance
    gi_filename = f'{config.model.code}_split{midx}_global_importance.csv'
    gi_df = pd.read_csv(split_output_dirs[midx] / gi_filename)
    gi_df = gi_df.rename(columns={'Unnamed: 0':'Feature'})
    gi_df = gi_df.rename(columns={'0':f'Split {midx}'})
    gi_dfs.append(gi_df)

    # Process Instance Folders
    cf = {}
    for qidx in ioi_df.ID.values:
        qidx_filename = f'{config.model.code}_split{midx}_local_cf.csv'        
        qidx_path = split_output_dirs[midx] / 'filtered_progressive' / str(qidx).zfill(3) / qidx_filename
        if not qidx_path.is_file():
            continue
        qidx_df = pd.read_csv(qidx_path)
        cf_count = qidx_df.shape[0]    
        cf[qidx] = {
            'Patient Index': qidx, 
            'CF Count': cf_count, 
            'Sparsity': np.nan if not cf_count else qidx_df['sparsity'].mean(),
            'L1 Mean': np.nan if not cf_count else qidx_df['L1_dist'].mean(),
            'L2 Mean': np.nan if not cf_count else qidx_df['L2_dist'].mean(),
            }
        
    # Counterfactual Stats for this Instance
    cf_df = pd.DataFrame(cf).T.reset_index(drop=True)
    cf_df['Patient Index'] = cf_df['Patient Index'].astype(int)
    cf_df['CF Count'] = cf_df['CF Count'].astype(int)
    cf_dfs[midx] = cf_df

    # Instance of Interest for this Model
    ioi[midx] = {
        'Model': f'Split {midx}', 
        'Instances': ioi_df.shape[0], 
        'Misclassified': ioi_df.misclassified.sum(),
        'Instances with CF': (cf_df['CF Count'] > 0).sum(),
        'Sparsity Mean': cf_df['Sparsity'].mean(),
        'L1 Mean': cf_df['L1 Mean'].mean(),
        'L2 Mean': cf_df['L2 Mean'].mean(),
        }    
    ioi_summary_dfs.append(pd.DataFrame(ioi).T)

## Concatenate results for models

# Global Importance
gi_summary_df = gi_dfs[0].merge(gi_dfs[1], on='Feature').merge(gi_dfs[2], on='Feature')
gi_summary_df['Mean'] = gi_summary_df[['Split 0', 'Split 1', 'Split 2']].mean(axis=1)
gi_summary_df = gi_summary_df.sort_values(by='Mean', ascending=False)

# Instance of Interest Summary
ioi_summary_df = pd.concat(ioi_summary_dfs, axis=0)

### Global Importance Summary

In [60]:
gi_summary_df

,Feature,Split 0,Split 1,Split 2,Mean
0,CMAPANK_L,1.000,1.000,1.000,1.000
1,CMAPKNE_L,1.000,0.998,1.000,0.999
2,CMAPKNE_R,1.000,1.000,0.996,0.999
4,CMAPANK_R,0.996,1.000,1.000,0.999
7,SSA_L,0.994,1.000,1.000,0.998
3,SPSA_L,0.998,0.998,0.996,0.997
5,FWAVE_L,0.996,0.996,0.993,0.995
6,FWAVE_R,0.996,0.994,0.995,0.995
11,SSA_R,0.988,0.996,0.998,0.994
9,MCV_L,0.988,0.994,0.995,0.992


### IOI Summary

In [61]:
ioi_summary_df

,Model,Instances,Misclassified,Instances with CF,Sparsity Mean,L1 Mean,L2 Mean
0,Split 0,6,4,4,23.7,107.113,30.823
1,Split 1,7,4,7,26.164,111.45,31.747
2,Split 2,9,6,6,24.81,99.698,29.54


### Model 0 Summary

In [62]:
cf_dfs[0] 

,Patient Index,CF Count,Sparsity,L1 Mean,L2 Mean
0,4,1,29.000,135.270,33.364
1,55,0,NaN,NaN,NaN
2,67,22,22.636,95.595,31.052
3,106,0,NaN,NaN,NaN
4,128,7,19.857,94.031,28.886
5,158,13,23.308,103.557,29.989


### Model 1 Summary

In [63]:
cf_dfs[1] 

,Patient Index,CF Count,Sparsity,L1 Mean,L2 Mean
0,12,1,27.00,97.340,28.436
1,54,8,28.75,129.708,35.250
2,73,10,29.40,142.819,36.453
3,94,13,17.00,70.046,26.089
4,119,16,26.00,100.251,29.720
5,142,4,30.00,124.140,32.598
6,178,5,25.00,115.844,33.682


### Model 2 Summary

In [64]:
cf_dfs[2] 

,Patient Index,CF Count,Sparsity,L1 Mean,L2 Mean
0,22,30,25.200,95.814,29.438
1,45,15,29.000,128.401,33.808
2,68,13,23.462,118.916,35.276
3,88,3,23.667,97.553,30.572
4,96,1,24.000,69.820,21.425
5,136,0,NaN,NaN,NaN
6,155,0,NaN,NaN,NaN
7,167,15,23.533,87.685,26.721


### Produce LaTeX Tables

#### Global Importances

In [65]:
print(gi_summary_df.to_latex(index=False))

\begin{tabular}{lrrrr}
\toprule
Feature & Split 0 & Split 1 & Split 2 & Mean \\
\midrule
CMAPANK_L & 1.000000 & 1.000000 & 1.000000 & 1.000000 \\
CMAPKNE_L & 1.000000 & 0.997877 & 1.000000 & 0.999292 \\
CMAPKNE_R & 1.000000 & 1.000000 & 0.996370 & 0.998790 \\
CMAPANK_R & 0.996078 & 1.000000 & 1.000000 & 0.998693 \\
SSA_L & 0.994118 & 1.000000 & 1.000000 & 0.998039 \\
SPSA_L & 0.998039 & 0.997877 & 0.996370 & 0.997429 \\
FWAVE_L & 0.996078 & 0.995754 & 0.992740 & 0.994858 \\
FWAVE_R & 0.996078 & 0.993631 & 0.994555 & 0.994755 \\
SSA_R & 0.988235 & 0.995754 & 0.998185 & 0.994058 \\
MCV_L & 0.988235 & 0.993631 & 0.994555 & 0.992140 \\
SPSA_R & 0.986275 & 0.989384 & 1.000000 & 0.991886 \\
MCV_R & 0.992157 & 0.993631 & 0.987296 & 0.991028 \\
SSC_R & 0.980392 & 0.989384 & 0.989111 & 0.986296 \\
SPSC_L & 0.988235 & 0.976645 & 0.985481 & 0.983454 \\
SSC_L & 0.976471 & 0.983015 & 0.985481 & 0.981655 \\
SPSC_R & 0.982353 & 0.976645 & 0.983666 & 0.980888 \\
FEET_MEAN_ESC & 0.978431 & 0.972399 & 0

#### Counterfactual Instances Summary

In [66]:
print(ioi_summary_df.to_latex(index=False))

\begin{tabular}{lllllll}
\toprule
Model & Instances & Misclassified & Instances with CF & Sparsity Mean & L1 Mean & L2 Mean \\
\midrule
Split 0 & 6 & 4 & 4 & 23.700300 & 107.113338 & 30.822752 \\
Split 1 & 7 & 4 & 7 & 26.164286 & 111.449701 & 31.746909 \\
Split 2 & 9 & 6 & 6 & 24.810256 & 99.698303 & 29.539751 \\
\bottomrule
\end{tabular}



#### Model 0 Summary

In [67]:
print(cf_dfs[0] .to_latex(index=False))

\begin{tabular}{rrrrr}
\toprule
Patient Index & CF Count & Sparsity & L1 Mean & L2 Mean \\
\midrule
4 & 1 & 29.000000 & 135.270000 & 33.364189 \\
55 & 0 & NaN & NaN & NaN \\
67 & 22 & 22.636364 & 95.595000 & 31.052419 \\
106 & 0 & NaN & NaN & NaN \\
128 & 7 & 19.857143 & 94.031429 & 28.885841 \\
158 & 13 & 23.307692 & 103.556923 & 29.988558 \\
\bottomrule
\end{tabular}



#### Model 1 Summary

In [68]:
print(cf_dfs[1] .to_latex(index=False))

\begin{tabular}{rrrrr}
\toprule
Patient Index & CF Count & Sparsity & L1 Mean & L2 Mean \\
\midrule
12 & 1 & 27.000000 & 97.340000 & 28.436209 \\
54 & 8 & 28.750000 & 129.707500 & 35.249737 \\
73 & 10 & 29.400000 & 142.819000 & 36.452940 \\
94 & 13 & 17.000000 & 70.046154 & 26.089443 \\
119 & 16 & 26.000000 & 100.251250 & 29.719526 \\
142 & 4 & 30.000000 & 124.140000 & 32.598221 \\
178 & 5 & 25.000000 & 115.844000 & 33.682288 \\
\bottomrule
\end{tabular}



#### Model 2 Summary

In [69]:
print(cf_dfs[2] .to_latex(index=False))

\begin{tabular}{rrrrr}
\toprule
Patient Index & CF Count & Sparsity & L1 Mean & L2 Mean \\
\midrule
22 & 30 & 25.200000 & 95.814333 & 29.437769 \\
45 & 15 & 29.000000 & 128.401333 & 33.807619 \\
68 & 13 & 23.461538 & 118.916154 & 35.276450 \\
88 & 3 & 23.666667 & 97.553333 & 30.571552 \\
96 & 1 & 24.000000 & 69.820000 & 21.424551 \\
136 & 0 & NaN & NaN & NaN \\
155 & 0 & NaN & NaN & NaN \\
167 & 15 & 23.533333 & 87.684667 & 26.720562 \\
\bottomrule
\end{tabular}

